In [ ]:
!pip install --upgrade  setuptools 

In [1]:
import gurobipy as gp
from gurobipy import GRB
from utils.data import *

In [2]:
start= dt.datetime(year=2000, month=1, day=1)
end= dt.datetime(year=2023, month=1, day=1)
all_stocks = ["NVDA", "AAPL", "BA", 'DB']

n = len(all_stocks)

pce = load_pce()
effr = load_ffr()

a = (0.005, 4)

mu = generate_mu(all_stocks, start, end)
Sigma = generate_Sigma(all_stocks, start, end)

new_sigma, sep_Sigma = add_covariates_to_covar(Sigma, all_stocks, [pce, effr], start, end)
new_mu, sep_mu = add_covariates_to_mu(mu, [pce, effr])

mu, Sigma = conditional_moments(sep_mu, sep_Sigma, a)

r = 0.05
T = 90
S = [yf.Ticker(s).history(period="1d") for s in all_stocks]

In [3]:
Sigma

array([[0.11562611, 0.03790047, 0.01591687, 0.0303147 ],
       [0.03790047, 0.04527312, 0.00398775, 0.01509983],
       [0.01591687, 0.00398775, 0.02974967, 0.02073798],
       [0.0303147 , 0.01509983, 0.02073798, 0.03747639]])

In [28]:
m = gp.Model()

S = [yf.Ticker(s).history(period="1d").Close.to_list()[0] for s in all_stocks]
sigma = [np.sqrt(Sigma[i, i]) for i in range(n)]

w = m.addMVar(n, lb=0, name="weight")

mu_d = m.addMVar(n, lb=-GRB.INFINITY, name="expectation")
Delta = m.addMVar(n, lb=0.1, ub=.5, name="Delta")
#Phi_inv = m.addMVar(n, lb=-GRB.INFINITY, name="phi of inverse CDF")
#phi = m.addMVar(n, lb=-GRB.INFINITY, name="phi")
#K = m.addMVar(n, lb=0, name="strike")
P = m.addMVar(n, lb=-GRB.INFINITY, name="put price")
#d1 = m.addMVar(n, lb=-GRB.INFINITY, name="d1")
#d2 = m.addMVar(n, lb=-GRB.INFINITY, name="d2")
var_norm = m.addVar(lb=0)

for i in range(n):
    Phi_inv_expr = 4/1.7*(Delta[i] - 0.5)
    K_expr = (1+sigma[i]*Phi_inv_expr)* S[i]
    d1_expr = (1 - K_expr/S[i] + T*(r + sigma[i]**2/2))/(sigma[i]*np.sqrt(T))
    d2_expr = d1_expr - sigma[i]*np.sqrt(T)
    cdf_d1_expr = 0.5 - (1/np.sqrt(2*np.pi))*d1_expr
    cdf_d2_expr = 0.5 - (1/np.sqrt(2*np.pi))*d2_expr
    P_expr = K_expr*np.exp(-r*T)*cdf_d2_expr - S[i]*cdf_d1_expr
    phi_expr = 1/np.sqrt(2*np.pi)*(1-Phi_inv_expr**2/2)


    m.addConstr(P[i] == P_expr)
    m.addConstr(mu_d[i] == mu[i] + sigma[i]*(Delta[i]*Phi_inv_expr+ phi_expr))

t = m.addVar(lb=-GRB.INFINITY)

m.addConstr(gp.quicksum([wi for wi in w]) == 1)
m.addConstr(gp.quicksum([mu_d[i]*w[i] for i in range(n)]) >= var_norm * t)

Sigma_d = np.zeros((n, n), dtype=gp.Var)

for i in range(n):
    for j in range(i+1, n):
        rho_ij = Sigma[i,j] / sigma[i]*sigma[j]
        sqrt_1mrho2 = np.sqrt(1 - rho_ij**2)

        eta_i = -4/1.7*(Delta[i] - 0.5)   
        eta_j = -4/1.7*(Delta[j] - 0.5)

        phi_i = 1/np.sqrt(2*np.pi)*(1 - eta_i**2/2)
        phi_j = 1/np.sqrt(2*np.pi)*(1 - eta_j**2/2)

        Phi_eta_i = 1 - Delta[i]
        Phi_eta_j = 1 - Delta[j]

        w_ij = (eta_i - rho_ij*eta_j) / sqrt_1mrho2
        w_ji = (eta_j - rho_ij*eta_i) / sqrt_1mrho2

        Phi_w_ij = 0.5 + 1/np.sqrt(2*np.pi)*w_ij
        Phi_w_ji = 0.5 + 1/np.sqrt(2*np.pi)*w_ji

        Phi2 = Phi_eta_i*Phi_eta_j + rho_ij*phi_i*phi_j

        phi2 = phi_i*phi_j / sqrt_1mrho2

        a_i = 4/1.7*(Delta[i] - 0.5)
        a_j = 4/1.7*(Delta[j] - 0.5)

       # lhs = m.addVar(lb=-GRB.INFINITY, name=f"EZZ_{i}_{j}_lhs")
        covar_ij = m.addVar(lb=-GRB.INFINITY, name=f"covar_{i}_{j}")

        #m.addConstr(lhs == EZZ_ij * Phi2)
        print(rho_ij)
        m.addConstr(
            covar_ij == mu[i]*mu[j]+rho_ij
            + (rho_ij * a_i * phi_i * Phi_w_ji
            + rho_ij * a_j * phi_j * Phi_w_ij
            + (1 - rho_ij**2) * phi2)/Phi2 -  mu_d[i]*mu_d[j]
        )

        Sigma_d[i, j] = covar_ij      
        Sigma_d[j, i] = covar_ij

    Phi_inv_i = 4/1.7 * (Delta[i] - 0.5)                          
    phi_i = 1/np.sqrt(2*np.pi) * (1 - Phi_inv_i**2 / 2)       
    
    var_i = m.addVar(lb=-GRB.INFINITY, name=f"var_{i}")

    m.addConstr(
        var_i == sigma[i]**2 *                                  \
        (Delta[i] * Phi_inv_i**2 *(1-Delta[i])                  \
        + phi_i*(-2*Delta[i]*Phi_inv_i + Phi_inv_i - phi_i)     \
        - Delta[i] + 1)
    )

    Sigma_d[i, i] = var_i


m.addConstr(var_norm == gp.quicksum([Sigma_d[i, i] * w[i]**2 for i in range(n)]) + \
                        2*gp.quicksum([Sigma_d[i, j] * w[i]*w[j] for i in range(n) for j in range(n) if j > i]))

m.setObjective(t, GRB.MAXIMIZE)

m.optimize()


#m.addConstrs(Phi_inv[i] == -1/1.7*nlfunc.log(1/Delta[i]-1) for i in range(n))
"""
m.addConstrs(Phi_inv[i] == 4/1.7*(Delta[i]-1/2) for i in range(n))

m.addConstrs(phi[i] == 1/np.sqrt(2*np.pi)*(1-Phi_inv[i]**2/2) for i in range(n))
m.addConstrs(K[i] == sigma[i]*Phi_inv[i] + S[i] for i in range(n))

m.addConstrs(d1[i] == (1-K[i]/S[i]+T*(r+sigma[i]**2/2))/(sigma[i]*np.sqrt(T)) for i in range(n))
m.addConstrs(d2[i] == d1[i] - sigma[i]*np.sqrt(T) for i in range(n))

m.addConstrs(cdf_d1[i] == 0.5 - (1/np.sqrt(2*np.pi))*d1[i] for i in range(n))
m.addConstrs(cdf_d2[i] == 0.5 - (1/np.sqrt(2*np.pi))*d2[i] for i in range(n))

m.addConstrs(P[i] == K[i]*np.exp(-r*T)*cdf_d2[i] - S[i]*cdf_d1[i] for i in range(n))

"""


0.023715749091279412
0.008073664281422714
0.01725854519411007
0.0032325728861822213
0.01373822627328831
0.0232757806981451
Gurobi Optimizer version 12.0.3 build v12.0.3rc0 (linux64 - "Fedora Linux 42 (Workstation Edition)")

CPU model: AMD Ryzen 9 5900X 12-Core Processor, instruction set [SSE2|AVX|AVX2]
Thread count: 12 physical cores, 24 logical processors, using up to 24 threads

Optimize a model with 1 rows, 28 columns and 4 nonzeros
Model fingerprint: 0x10c750a0
Model has 9 quadratic constraints
Model has 11 general nonlinear constraints (138 nonlinear terms)
Variable types: 28 continuous, 0 integer (0 binary)
Coefficient statistics:
  Matrix range     [1e+00, 1e+00]
  QMatrix range    [2e-02, 1e+00]
  QLMatrix range   [1e-02, 3e+01]
  Objective range  [1e+00, 1e+00]
  Bounds range     [1e-01, 5e-01]
  RHS range        [1e+00, 1e+00]
  QRHS range       [1e-02, 2e+02]
Presolve model has 11 nlconstr
Added 152 variables to disaggregate expressions.
Presolve removed 0 rows and 4 column

'\nm.addConstrs(Phi_inv[i] == 4/1.7*(Delta[i]-1/2) for i in range(n))\n\nm.addConstrs(phi[i] == 1/np.sqrt(2*np.pi)*(1-Phi_inv[i]**2/2) for i in range(n))\nm.addConstrs(K[i] == sigma[i]*Phi_inv[i] + S[i] for i in range(n))\n\nm.addConstrs(d1[i] == (1-K[i]/S[i]+T*(r+sigma[i]**2/2))/(sigma[i]*np.sqrt(T)) for i in range(n))\nm.addConstrs(d2[i] == d1[i] - sigma[i]*np.sqrt(T) for i in range(n))\n\nm.addConstrs(cdf_d1[i] == 0.5 - (1/np.sqrt(2*np.pi))*d1[i] for i in range(n))\nm.addConstrs(cdf_d2[i] == 0.5 - (1/np.sqrt(2*np.pi))*d2[i] for i in range(n))\n\nm.addConstrs(P[i] == K[i]*np.exp(-r*T)*cdf_d2[i] - S[i]*cdf_d1[i] for i in range(n))\n\n'

In [29]:
weights = np.zeros(n)

var = m.getVars()

weights = [v.X for v in var if "weight" in v.VarName]

deltas = [v.X for v in var if "Delta" in v.VarName]
P = [v.X for v in var if "price" in v.VarName]
mud = [v.X for v in var if "expectation" in v.VarName]
weights, deltas, P, mud

([0.0, 0.0, 1.0, 0.0],
 [0.1902751566862238, 0.4051425415971061, 0.5, 0.1516266429328577],
 [146.79064518158535,
  219.13731437926756,
  215.5402269736596,
  27.527723077734237],
 [0.17032879109853613,
  0.13214339398381394,
  0.10316620408579491,
  0.016762025745067067])

In [27]:
i = 0
deltas[0], S[0], sigma[i]

(0.1902751566862238, 199.63999938964844, np.float64(0.3400383903582107))

In [47]:
i = 0

Phi_inv_expr = 4/1.7*(deltas[i] - 0.5)
K_expr = (sigma[i]*Phi_inv_expr + 1) * S[i]
d1_expr = (1 - K_expr/S[i] + T*(r + sigma[i]**2/2))/(sigma[i]*np.sqrt(T))
d2_expr = d1_expr - sigma[i]*np.sqrt(T)
cdf_d1_expr = 0.5 - (1/np.sqrt(2*np.pi))*d1_expr
cdf_d2_expr = 0.5 - (1/np.sqrt(2*np.pi))*d2_expr
P_expr = K_expr*np.exp(-r*T)*cdf_d2_expr - S[i]*cdf_d1_expr

K_expr, d2_expr

(np.float64(150.16763993268583), np.float64(-0.14116021433634485))

In [12]:
vars = m.getVars()

var = np.array([v for v in vars if  v.VarName[:4] == "var_"])
covar = np.array([v for v in vars if  v.VarName[:6] == "covar_"])

Sigma_result = np.zeros((n, n), dtype=gp.Var)

for i in range(n):
    Sigma_result[i, i] = np.array([v.X for v in vars if v.VarName == f"var_{i}"])[0]
    for j in range(i+1, n):
        print([v for v in vars if v.VarName == f"covar_{i}_{j}"])
        Sigma_result[i, j] = np.array([v.X for v in vars if v.VarName == f"covar_{i}_{j}"])[0]
        Sigma_result[j, i] = np.array([v.X for v in vars if v.VarName == f"covar_{i}_{j}"])[0]

Sigma_result

[<gurobi.Var covar_0_1 (value -30473.4225965852)>]
[<gurobi.Var covar_0_2 (value -30331.310360756193)>]
[<gurobi.Var covar_0_3 (value -3727.2328326270044)>]
[<gurobi.Var covar_1_2 (value -46629.962867753115)>]
[<gurobi.Var covar_1_3 (value -5730.147368283041)>]
[<gurobi.Var covar_2_3 (value -5703.414691382499)>]


array([[np.float64(0.039410586992324775), np.float64(-30473.4225965852),
        np.float64(-30331.310360756193), np.float64(-3727.2328326270044)],
       [np.float64(-30473.4225965852), np.float64(0.01543111916647241),
        np.float64(-46629.962867753115), np.float64(-5730.147368283041)],
       [np.float64(-30331.310360756193), np.float64(-46629.962867753115),
        np.float64(0.010140026272257418), np.float64(-5703.414691382499)],
       [np.float64(-3727.2328326270044), np.float64(-5730.147368283041),
        np.float64(-5703.414691382499), np.float64(0.02859407683296517)]],
      dtype=object)

In [18]:
import plotly.express as px

px.line(benchmark(all_stocks, weights, "SPY", end, dt.datetime.today()))

In [ ]:
px.line(benchmark(all_stocks, np.ones(len(all_stocks))/len(all_stocks), "SPY", end, dt.datetime.today()))

In [ ]:
sigma = [np.std(get_hist(s, start, end).Open.to_numpy()) for s in all_stocks]

sigma

In [ ]:
import plotly.express as px
import sympy as sp

K = sp.Symbol("K")
S = S[0]
sigma = sigma[0]

d1 = (1-K/S+T*(r+sigma**2/2))/(sigma*np.sqrt(T)) 

sp.plot(d1)

In [ ]:
#if m.status == GRB.INFEASIBLE:
m.computeIIS()
m.write("model.ilp")
for c in m.getConstrs():
    if c.IISConstr:
        print(f"Infeasible constraint: {c.ConstrName}")
for v in m.getVars():
    if v.IISLB:
        print(f"Infeasible lower bound: {v.VarName} lb={v.LB}")
    if v.IISUB:
        print(f"Infeasible upper bound: {v.VarName} ub={v.UB}")
    
